In [1]:
import pandas as pd
import numpy as np
from CCA_utils import *

## Baseline Model Panel

In [10]:
study_sovereigns = [
    'Saudi Arabia', 'UAE (Abu Dhabi)', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia','Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand']

cca_panel_df = pd.read_csv('../data/processed/CCA_V2/CCA_panel.csv')
cca_panel_df['date'] = pd.to_datetime(cca_panel_df['date'])
cca_panel_df = cca_panel_df[cca_panel_df['country'].isin(study_sovereigns)].copy()

T = 5.0
vol_window = 52
freq = 'W'

cca_panel_df.set_index(['date','country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)

cca_panel_df['domestic_debt_bn_local'] = 0

## M2 Specific Data

In [11]:
ovx_df = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv')
ovx_df['date'] = pd.to_datetime(ovx_df['date'])
ovx_df = ovx_df.sort_values('date')
ovx_df['OVXCLS'] = ovx_df['OVXCLS'].ffill()

cca_panel_df = cca_panel_df.reset_index()
cca_panel_df = cca_panel_df.sort_values('date')

cca_panel_df = pd.merge_asof(
    cca_panel_df,
    ovx_df[['date', 'OVXCLS']],
    on='date',
    direction='backward'
)

cca_panel_df.set_index(['date', 'country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)
print(f"OVX NAs: {cca_panel_df['OVXCLS'].isna().sum()}")

OVX NAs: 0


## M2 Parameters & Pricer

In [12]:
# --- Sigmoid: OVX -> annualized jump intensity ---
# lambda(OVX) = L / (1 + exp(-k * (OVX - x0)))
L_sig  = 0.2868     # FILL: max daily jump frequency from sigmoid fit
k_sig  = 0.1117    # FILL: steepness from sigmoid fit
x0_sig = 60    # FILL: midpoint OVX from sigmoid fit

def ovx_to_lambda(ovx):
    """OVX -> annualized jump intensity"""
    if np.isnan(ovx):
        return 0.0
    lam_daily = L_sig / (1.0 + np.exp(-k_sig * (ovx - x0_sig)))
    return 52 * lam_daily

# --- Fixed jump size ---
gamma = 0.10
J = -0.10

# --- Initialize pricer ---
pricer = FixedJumpCCAPricer(J=J*gamma, max_terms=80)

## Run M2

In [13]:
results = pd.DataFrame()

print("Starting M2 calibration...")

for country, group in cca_panel_df.groupby('country'):
    print(f"Processing {country}...")
    df = group.copy().sort_values('date').reset_index(drop=True)

    r_d = df['domestic_rate']
    r_f = df['risk_free_rate']
    M_bn = df['monetary_base_bn_local']
    dom_D_bn = df['domestic_debt_bn_local']
    ext_D_bn = df['external_debt_bn_usd']
    fx_rate = df['fx_rate']

    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(M_bn, dom_D_bn, fx_rate, r_d, r_f)
    ]

    ann_factor = np.sqrt(52) if freq == 'W' else np.sqrt(12)
    log_ret = np.log(df['LCL_usd'] / df['LCL_usd'].shift(1))
    df['sigma_lcl'] = log_ret.rolling(window=vol_window).std() * ann_factor

    df['B_f'] = [
        compute_barrier_kvm(debt, rf, T)
        for debt, rf in zip(ext_D_bn, r_f)
    ]
    
    df['lambda_annual'] = df['OVXCLS'].apply(ovx_to_lambda)

    out = {'implied_V': [], 'implied_sigma_V': [], 'cca_converged': []}

    for i, row in df.iterrows():
        lam = row['lambda_annual']

        cca_base = solve_CCA(row['LCL_usd'], row['sigma_lcl'], row['B_f'], r_f.iloc[i], T)
        v_guess   = cca_base['V']       if cca_base['converged'] else (row['LCL_usd'] + row['B_f'])
        sig_guess = cca_base['sigma_V'] if cca_base['converged'] else (row['sigma_lcl'] * row['LCL_usd'] / (row['LCL_usd'] + row['B_f']))

        cca = pricer.solve_CCA(row['LCL_usd'], row['sigma_lcl'], row['B_f'],
                               r_f.iloc[i], T, lam,
                               v_guess=v_guess, sig_guess=sig_guess)

        out['implied_V'].append(cca['V'])
        out['implied_sigma_V'].append(cca.get('sigma_total', cca.get('sigma_diff', np.nan)))
        out['cca_converged'].append(cca['converged'])

    for col, vals in out.items():
        df[col] = vals

    results = pd.concat([results, df])

START_DATE = '2015-01-01'
END_DATE = '2024-12-31'
results = results[
    (results['date'] >= START_DATE) & (results['date'] <= END_DATE)
].copy()

print("Calibration complete!")

Starting M2 calibration...
Processing Brazil...
Processing Chile...
Processing China...
Processing Colombia...
Processing Egypt...
Processing Indonesia...
Processing Malaysia...
Processing Mexico...
Processing Philippines...
Processing Qatar...
Processing Saudi Arabia...
Processing South Africa...
Processing South Korea...
Processing Thailand...
Processing Turkey...
Processing UAE (Abu Dhabi)...
Calibration complete!


In [14]:
results[['date', 'country', 'cds_spread', 'risk_free_rate',
         'implied_V', 'implied_sigma_V', 'cca_converged',
         'B_f', 'LCL_usd', 'sigma_lcl',
         'lambda_annual']].to_csv(
    '../output/results/M2_results_5YCDS_weekly.csv', index=False)

In [38]:
print(results['lambda_annual'].describe())
print(results['OVXCLS'].describe())

count    1920.000000
mean        2.206214
std         2.833772
min         0.195638
25%         0.542493
50%         1.187001
75%         2.574523
max        14.913535
Name: lambda_annual, dtype: float64
count    1920.000000
mean       40.596083
std        17.073944
min        21.320000
25%        30.662500
50%        38.085000
75%        45.970000
max       170.550000
Name: OVXCLS, dtype: float64


In [39]:
print(type(pricer))
print(pricer.J)
print(pricer.k)

<class 'CCA_utils.FixedJumpCCAPricer'>
-0.010000000000000002
-0.010000000000000002


In [23]:
# Pick a 2020 row with high OVX
row = results[results['OVXCLS'] > 80].iloc[0]
lam = ovx_to_lambda(row['OVXCLS'])
print(f"OVX: {row['OVXCLS']:.1f}, lambda: {lam:.2f}")

# M0 solve
cca_m0 = solve_CCA(row['LCL_usd'], row['sigma_lcl'], row['B_f'], row['risk_free_rate'], T)
risk_m0 = compute_risk(cca_m0['V'], cca_m0['sigma_V'], row['B_f'], row['risk_free_rate'], T)

# M2 solve  
cca_m2 = pricer.solve_CCA(row['LCL_usd'], row['sigma_lcl'], row['B_f'], row['risk_free_rate'], T, lam)
risk_m2 = pricer.compute_risk(cca_m2['V'], cca_m2['sigma_diff'], row['B_f'], row['risk_free_rate'], T, lam)

print(f"\nM0: V={cca_m0['V']:.2f}  sigma={cca_m0['sigma_V']:.4f}  spread={risk_m0['credit_spread_bps']:.2f}")
print(f"M2: V={cca_m2['V']:.2f}  sigma={cca_m2['sigma_total']:.4f}  spread={risk_m2['credit_spread_bps']:.2f}")

OVX: 118.5, lambda: 97.26

M0: V=1074.09  sigma=0.0717  spread=0.00
M2: V=1123.72  sigma=1.0413  spread=-0.00


In [24]:
from math import factorial
lam_prime = 97.26 * (1 + (-0.10))
lam_T = lam_prime * 5
print(f"lambda' * T = {lam_T:.1f}")
total_weight = sum(np.exp(-lam_T) * lam_T**n / factorial(n) for n in range(21))
print(f"Weight captured by 20 terms: {total_weight:.2e}")

lambda' * T = 437.7
Weight captured by 20 terms: 2.40e-156


In [9]:
scenarios = [
    # --- λ sweep (baseline) ---
    (600, 0.20, 800, 0.025, 5, 0,     "λ=0 (M0)"),
    (600, 0.20, 800, 0.025, 5, 0.5,   "λ=0.5"),
    (600, 0.20, 800, 0.025, 5, 1,     "λ=1"),
    (600, 0.20, 800, 0.025, 5, 2,     "λ=2"),
    (600, 0.20, 800, 0.025, 5, 5,     "λ=5"),
    (600, 0.20, 800, 0.025, 5, 10,    "λ=10"),
    (600, 0.20, 800, 0.025, 5, 15,    "λ=15"),
    (600, 0.20, 800, 0.025, 5, 20,    "λ=20"),
    (600, 0.20, 800, 0.025, 5, 30,    "λ=30"),
    (600, 0.20, 800, 0.025, 5, 50,    "λ=50"),
    (600, 0.20, 800, 0.025, 5, 100,   "λ=100"),
    # --- T sweep ---
    (600, 0.20, 800, 0.025, 1, 5,     "T=1, λ=5"),
    (600, 0.20, 800, 0.025, 1, 20,    "T=1, λ=20"),
    (600, 0.20, 800, 0.025, 1, 50,    "T=1, λ=50"),
    (600, 0.20, 800, 0.025, 10, 5,    "T=10, λ=5"),
    (600, 0.20, 800, 0.025, 10, 20,   "T=10, λ=20"),
    # --- Leverage sweep ---
    (100, 0.20, 800, 0.025, 5, 5,     "LCL=100, λ=5"),
    (400, 0.20, 800, 0.025, 5, 5,     "LCL=400, λ=5"),
    (800, 0.20, 800, 0.025, 5, 5,     "LCL=800, λ=5"),
    (1500, 0.20, 800, 0.025, 5, 5,    "LCL=1500, λ=5"),
    (100, 0.20, 800, 0.025, 5, 20,    "LCL=100, λ=20"),
    (1500, 0.20, 800, 0.025, 5, 20,   "LCL=1500, λ=20"),
    # --- Vol sweep ---
    (600, 0.02, 800, 0.025, 5, 5,     "σ=0.02, λ=5"),
    (600, 0.05, 800, 0.025, 5, 5,     "σ=0.05, λ=5"),
    (600, 0.10, 800, 0.025, 5, 5,     "σ=0.10, λ=5"),
    (600, 0.30, 800, 0.025, 5, 5,     "σ=0.30, λ=5"),
    (600, 0.50, 800, 0.025, 5, 5,     "σ=0.50, λ=5"),
    (600, 0.02, 800, 0.025, 5, 20,    "σ=0.02, λ=20"),
    (600, 0.50, 800, 0.025, 5, 20,    "σ=0.50, λ=20"),
    # --- Rate sweep ---
    (600, 0.20, 800, 0.00, 5, 5,      "r=0, λ=5"),
    (600, 0.20, 800, 0.05, 5, 5,      "r=5%, λ=5"),
    (600, 0.20, 800, 0.10, 5, 5,      "r=10%, λ=5"),
    # --- GCC-like: huge LCL, tiny vol ---
    (2000, 0.02, 200, 0.025, 5, 5,    "GCC: big LCL, low vol, λ=5"),
    (2000, 0.02, 200, 0.025, 5, 20,   "GCC: big LCL, low vol, λ=20"),
    # --- Egypt-like: high vol, high leverage ---
    (200, 0.30, 600, 0.025, 5, 5,     "Egypt-like, λ=5"),
    (200, 0.30, 600, 0.025, 5, 20,    "Egypt-like, λ=20"),
    # --- Edge cases ---
    (600, 0.20, 800, 0.025, 5, 0.01,  "λ=0.01 (near zero)"),
    (600, 0.20, 600, 0.025, 5, 5,     "LCL=B_f, λ=5"),
    (800, 0.20, 600, 0.025, 5, 5,     "LCL>B_f, λ=5"),
    (10, 0.20, 800, 0.025, 5, 5,      "tiny LCL, λ=5"),
]

print(f"{'Scenario':<30} {'V_M0':>8} {'V_M2':>8} {'σ_M0':>7} {'σ_M2':>7} {'DD_M0':>7} {'DD_M2':>7} {'sprd_M0':>8} {'sprd_M2':>8} {'put_M2':>8} {'lev_M2':>7} {'conv':>5}")
print("-" * 135)

for LCL, sig_lcl, Bf, rf, T, lam, label in scenarios:
    cca_m0 = solve_CCA(LCL, sig_lcl, Bf, rf, T)
    risk_m0 = compute_risk(cca_m0['V'], cca_m0['sigma_V'], Bf, rf, T)

    cca_m2 = pricer.solve_CCA(LCL, sig_lcl, Bf, rf, T, lam)
    risk_m2 = pricer.compute_risk(cca_m2['V'], cca_m2['sigma_diff'], Bf, rf, T, lam)

    v0 = cca_m0['V'] if cca_m0['converged'] else np.nan
    v2 = cca_m2['V'] if cca_m2.get('converged') else np.nan
    s0 = cca_m0['sigma_V'] if cca_m0['converged'] else np.nan
    s2 = risk_m2.get('sigma_total', np.nan)
    d0 = risk_m0.get('d2', np.nan)
    d2 = risk_m2.get('d2', np.nan)
    sp0 = risk_m0.get('credit_spread_bps', np.nan)
    sp2 = risk_m2.get('credit_spread_bps', np.nan)
    put2 = risk_m2.get('put_value', np.nan)
    lev2 = risk_m2.get('leverage', np.nan)
    conv = 'Y' if cca_m2.get('converged') else 'N'

    print(f"{label:<30} {v0:>8.1f} {v2:>8.1f} {s0:>7.4f} {s2:>7.4f} {d0:>7.3f} {d2:>7.3f} {sp0:>8.2f} {sp2:>8.2f} {put2:>8.2f} {lev2:>7.4f} {conv:>5}")

Scenario                           V_M0     V_M2    σ_M0    σ_M2   DD_M0   DD_M2  sprd_M0  sprd_M2   put_M2  lev_M2  conv
---------------------------------------------------------------------------------------------------------------------------------------
λ=0 (M0)                         1305.9   1305.9  0.0920  0.0920   2.888   3.695     0.22     0.22     0.08  0.6126     Y
λ=0.5                            1305.9   1305.7  0.0920  0.0921   2.888   3.351     0.22     0.81     0.28  0.6127     Y
λ=1                              1305.9   1303.3  0.0920  0.1054   2.888   2.992     0.22     2.77     0.98  0.6138     Y
λ=2                              1305.9   1287.6  0.0920  0.1494   2.888   2.381     0.22    17.35     6.10  0.6213     Y
λ=5                              1305.9   1223.9  0.0920  0.2356   2.888   1.618     0.22   108.54    37.29  0.6537     Y
λ=10                             1305.9   1132.4  0.0920  0.3332   2.888   1.074     0.22   305.09    99.88  0.7065     Y
λ=15      

In [10]:
pricer = FixedJumpCCAPricer(J=-0.10, max_terms=80)

def ovx_to_lambda(ovx):
    if np.isnan(ovx):
        return 0.0
    lam_w = 0.2868 / (1 + np.exp(-0.1117 * (ovx - 60)))
    return lam_w * 52

scenarios = [
    (600, 0.20, 800, 0.025, 5, 0,                  "λ=0 (M0)"),
    (600, 0.20, 800, 0.025, 5, ovx_to_lambda(20),  "OVX=20 (calm)"),
    (600, 0.20, 800, 0.025, 5, ovx_to_lambda(36),  "OVX=36 (median)"),
    (600, 0.20, 800, 0.025, 5, ovx_to_lambda(45),  "OVX=45"),
    (600, 0.20, 800, 0.025, 5, ovx_to_lambda(60),  "OVX=60 (stress)"),
    (600, 0.20, 800, 0.025, 5, ovx_to_lambda(80),  "OVX=80"),
    (600, 0.20, 800, 0.025, 5, ovx_to_lambda(100), "OVX=100"),
    (600, 0.20, 800, 0.025, 5, ovx_to_lambda(120), "OVX=120 (COVID)"),
]

print(f"{'Scenario':<25} {'λ_ann':>7} {'V_M2':>8} {'σ_M2':>7} {'DD_M2':>7} {'sprd_M2':>8} {'conv':>5}")
print("-" * 75)

for LCL, sig_lcl, Bf, rf, T, lam, label in scenarios:
    cca_m2 = pricer.solve_CCA(LCL, sig_lcl, Bf, rf, T, lam)
    risk_m2 = pricer.compute_risk(cca_m2['V'], cca_m2['sigma_diff'], Bf, rf, T, lam)
    print(f"{label:<25} {lam:>7.1f} {cca_m2['V']:>8.1f} {risk_m2.get('sigma_total',np.nan):>7.4f} "
          f"{risk_m2.get('d2',np.nan):>7.3f} {risk_m2.get('credit_spread_bps',np.nan):>8.2f} "
          f"{'Y' if cca_m2.get('converged') else 'N':>5}")

Scenario                    λ_ann     V_M2    σ_M2   DD_M2  sprd_M2  conv
---------------------------------------------------------------------------
λ=0 (M0)                      0.0   1305.9  0.0920   3.695     0.22     Y
OVX=20 (calm)                 0.2   1305.9  0.0920   3.534     0.41     Y
OVX=36 (median)               1.0   1303.8  0.1030   3.039     2.37     Y
OVX=45                        2.4   1284.6  0.1616   2.250    24.60     Y
OVX=60 (stress)               7.5   1166.9  0.2883   1.287   208.74     Y
OVX=80                       13.5   1076.6  0.3867   0.843   445.06     Y
OVX=100                      14.7   1058.9  0.4046   0.774   495.49     Y
OVX=120 (COVID)              14.9      nan     nan     nan      nan     N


In [11]:
# Debug OVX=120
lam = ovx_to_lambda(120)
print(f"λ = {lam:.4f}")

# Try with explicit warm-start from the OVX=100 solution
cca_100 = pricer.solve_CCA(600, 0.20, 800, 0.025, 5, ovx_to_lambda(100))
cca_120 = pricer.solve_CCA(600, 0.20, 800, 0.025, 5, lam,
                            v_guess=cca_100['V'], sig_guess=cca_100['sigma_diff'])
print(f"With warm-start: converged={cca_120['converged']}, V={cca_120['V']:.1f}")

λ = 14.8953
With warm-start: converged=True, V=1054.5
